In [24]:
import pandas as pd
import os
import os.path as op
import glob
import re

# Find all .nii.gz files in habenula-rois directory
rois_dir = "/Users/chloehampson/Desktop/habenula-rois"
nii_files = glob.glob(op.join(rois_dir, "*.nii.gz"))

# Extract subject IDs from filenames
subject_ids = []
for nii_file in nii_files:
    filename = op.basename(nii_file)
    # Extract subject ID using regex (matches sub-XXXXX pattern)
    match = re.match(r'(sub-\d+)_', filename)
    if match:
        subject_ids.append(match.group(1))

# Create DataFrame and save to TSV
drawn_subjects = sorted(set(subject_ids))  # Remove duplicates and sort
drawn_df = pd.DataFrame({'subject': drawn_subjects})

output_path = op.join(rois_dir, "drawn-habenulas.tsv")
drawn_df.to_csv(output_path, sep='\t', index=False)

print(f"Found {len(drawn_subjects)} unique subjects with drawn habenulas")
print(f"Saved to: {output_path}")
print(f"\nFirst 10 subjects:")
print(drawn_subjects[:10])

# Load habenula-rois_subjects.tsv for comparison
rois_path = op.join(rois_dir, "habenula-rois_subjects.tsv")
rois_df = pd.read_csv(rois_path, sep='\t')
rois_subjects = rois_df['subject'].values

print(f"\nROIs subjects: {len(rois_subjects)} subjects")

Found 1482 unique subjects with drawn habenulas
Saved to: /Users/chloehampson/Desktop/habenula-rois/drawn-habenulas.tsv

First 10 subjects:
['sub-0050004', 'sub-0050005', 'sub-0050006', 'sub-0050007', 'sub-0050008', 'sub-0050010', 'sub-0050011', 'sub-0050012', 'sub-0050013', 'sub-0050014']

ROIs subjects: 1483 subjects


In [25]:
# Extract subjects from the original table file
table_path = op.join(rois_dir, "sub-group_task-rest_desc-1S2StTesthabenula_table.txt")

# Read the table file
with open(table_path, 'r') as f:
    lines = f.readlines()

# Find the header line and extract subjects
original_subjects = []
for line in lines:
    # Skip comment lines and empty lines
    if line.strip() and not line.startswith('#'):
        # Split by whitespace and get the first column (Subj)
        parts = line.split()
        if parts and parts[0] != 'Subj':  # Skip header row if present
            subj = parts[0]
            # Only add if it looks like a subject ID
            if subj.startswith('sub-'):
                original_subjects.append(subj)

# Remove duplicates and sort
original_subjects = sorted(set(original_subjects))

# Save to TSV
original_tsv_path = op.join(rois_dir, "original-subjects.tsv")
original_df = pd.DataFrame({'subject': original_subjects})
original_df.to_csv(original_tsv_path, sep='\t', index=False)

print(f"Found {len(original_subjects)} unique subjects in original table")
print(f"Saved to: {original_tsv_path}")
print(f"\nFirst 10 subjects:")
print(original_subjects[:10])

Found 1584 unique subjects in original table
Saved to: /Users/chloehampson/Desktop/habenula-rois/original-subjects.tsv

First 10 subjects:
['sub-0050004', 'sub-0050005', 'sub-0050006', 'sub-0050007', 'sub-0050008', 'sub-0050010', 'sub-0050011', 'sub-0050012', 'sub-0050013', 'sub-0050014']


In [26]:
# Compare the drawn subjects to original subjects
# Find subjects in drawn habenulas that are NOT in original
drawn_only = set(drawn_subjects) - set(original_subjects)
original_only = set(original_subjects) - set(drawn_subjects)

print(f"Subjects with drawn habenulas but NOT in original table: {len(drawn_only)}")
print(sorted(drawn_only))

print(f"\nSubjects in original table but NO drawn habenulas: {len(original_only)}")
print(sorted(original_only))

# Show overlap
overlap = set(drawn_subjects) & set(original_subjects)
print(f"\nSubjects in both: {len(overlap)}")

Subjects with drawn habenulas but NOT in original table: 0
[]

Subjects in original table but NO drawn habenulas: 102
['sub-0050282', 'sub-0050332', 'sub-0050345', 'sub-0050354', 'sub-0050360', 'sub-0050570', 'sub-0050694', 'sub-0050702', 'sub-0050818', 'sub-0051000', 'sub-0051008', 'sub-28693', 'sub-28799', 'sub-28974', 'sub-29104', 'sub-29443', 'sub-29617', 'sub-29628', 'sub-29867', 'sub-29868', 'sub-29869', 'sub-29870', 'sub-29872', 'sub-29873', 'sub-29874', 'sub-29875', 'sub-29876', 'sub-29877', 'sub-29878', 'sub-29879', 'sub-29881', 'sub-29882', 'sub-29883', 'sub-29885', 'sub-29886', 'sub-29888', 'sub-29891', 'sub-29892', 'sub-29894', 'sub-29895', 'sub-29896', 'sub-29897', 'sub-29899', 'sub-29900', 'sub-29901', 'sub-29904', 'sub-29905', 'sub-29906', 'sub-29907', 'sub-29908', 'sub-29911', 'sub-29912', 'sub-29913', 'sub-29915', 'sub-29916', 'sub-29917', 'sub-29997', 'sub-29998', 'sub-30000', 'sub-30001', 'sub-30004', 'sub-30005', 'sub-30006', 'sub-30007', 'sub-30008', 'sub-30009', '

In [27]:
# Show subjects with drawn habenulas not in original table
print(f"Subjects with drawn habenulas but NOT in original table ({len(drawn_only)} subjects):\n")
print(sorted(drawn_only))

Subjects with drawn habenulas but NOT in original table (0 subjects):

[]


In [28]:
# Save drawn_only subjects to CSV (subjects with drawings but not in original table)
output_csv_path = "/Users/chloehampson/Desktop/habenula-rois/drawn_not_in_original.csv"
drawn_only_df = pd.DataFrame({'subject': sorted(drawn_only)})
drawn_only_df.to_csv(output_csv_path, index=False)
print(f"Saved drawn-only subjects to: {output_csv_path}")

Saved drawn-only subjects to: /Users/chloehampson/Desktop/habenula-rois/drawn_not_in_original.csv


In [29]:
# Compare group-participants.tsv to drawn habenulas
participants_path = op.join(rois_dir, "group-participants.tsv")
participants_df = pd.read_csv(participants_path, sep='\t')
participants_subjects = participants_df['participant_id'].values

print(f"Found {len(participants_subjects)} subjects in group-participants.tsv")

# Compare participants to drawn habenulas
participants_only = set(participants_subjects) - set(drawn_subjects)
drawn_not_in_participants = set(drawn_subjects) - set(participants_subjects)

print(f"\nSubjects in group-participants but NO drawn habenulas: {len(participants_only)}")
print(sorted(participants_only)[:20])  # Show first 20

print(f"\nSubjects with drawn habenulas but NOT in group-participants: {len(drawn_not_in_participants)}")
print(sorted(drawn_not_in_participants))

# Show overlap
overlap_participants = set(drawn_subjects) & set(participants_subjects)
print(f"\nSubjects in both: {len(overlap_participants)}")

Found 1584 subjects in group-participants.tsv

Subjects in group-participants but NO drawn habenulas: 102
['sub-0050282', 'sub-0050332', 'sub-0050345', 'sub-0050354', 'sub-0050360', 'sub-0050570', 'sub-0050694', 'sub-0050702', 'sub-0050818', 'sub-0051000', 'sub-0051008', 'sub-28693', 'sub-28799', 'sub-28974', 'sub-29104', 'sub-29443', 'sub-29617', 'sub-29628', 'sub-29867', 'sub-29868']

Subjects with drawn habenulas but NOT in group-participants: 0
[]

Subjects in both: 1482


In [30]:
# Update habenula-rois_subjects.tsv with filename and subject_id
rois_data = []
for nii_file in nii_files:
    filename = op.basename(nii_file)
    # Extract subject ID using regex
    match = re.match(r'(sub-\d+)_', filename)
    if match:
        subject_id = match.group(1)
        rois_data.append({'filename': filename, 'subject': subject_id})

# Create DataFrame with filename and subject columns
updated_rois_df = pd.DataFrame(rois_data)
updated_rois_df = updated_rois_df.sort_values('subject').reset_index(drop=True)

# Save updated TSV
updated_rois_path = op.join(rois_dir, "habenula-rois_subjects.tsv")
updated_rois_df.to_csv(updated_rois_path, sep='\t', index=False)

print(f"Updated habenula-rois_subjects.tsv with {len(updated_rois_df)} subjects")
print(f"Saved to: {updated_rois_path}")
print(f"\nFirst 10 entries:")
print(updated_rois_df.head(10))

Updated habenula-rois_subjects.tsv with 1483 subjects
Saved to: /Users/chloehampson/Desktop/habenula-rois/habenula-rois_subjects.tsv

First 10 entries:
                                            filename      subject
0  sub-0050004_space-MNI152NLin2009cAsym_res-2_de...  sub-0050004
1  sub-0050005_space-MNI152NLin2009cAsym_res-2_de...  sub-0050005
2  sub-0050006_space-MNI152NLin2009cAsym_res-2_de...  sub-0050006
3  sub-0050007_space-MNI152NLin2009cAsym_res-2_de...  sub-0050007
4  sub-0050008_space-MNI152NLin2009cAsym_res-2_de...  sub-0050008
5  sub-0050010_space-MNI152NLin2009cAsym_res-2_de...  sub-0050010
6  sub-0050011_space-MNI152NLin2009cAsym_res-2_de...  sub-0050011
7  sub-0050012_space-MNI152NLin2009cAsym_res-2_de...  sub-0050012
8  sub-0050013_space-MNI152NLin2009cAsym_res-2_de...  sub-0050013
9  sub-0050014_space-MNI152NLin2009cAsym_res-2_de...  sub-0050014


In [31]:
# Extract subject IDs from sub-group_task-rest_space-MNI152NLin2009cAsym_briks.txt
briks_txt_path = op.join(rois_dir, "sub-group_task-rest_space-MNI152NLin2009cAsym_briks.txt")

# Read the file and extract subject IDs from file paths
briks_subjects = []
with open(briks_txt_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            # Extract filename from path
            filename = op.basename(line)
            # Extract subject ID (first part before underscore)
            match = re.match(r'(sub-\d+)_', filename)
            if match:
                briks_subjects.append(match.group(1))

# Remove duplicates and sort
briks_subjects = sorted(set(briks_subjects))

print(f"Found {len(briks_subjects)} unique subjects in briks.txt")
print(f"\nFirst 10 subjects:")
print(briks_subjects[:10])

# Compare to habenula-rois subjects (from updated_rois_df)
rois_subjects_set = set(updated_rois_df['subject'].values)
briks_subjects_set = set(briks_subjects)

briks_only = briks_subjects_set - rois_subjects_set
rois_only = rois_subjects_set - briks_subjects_set
overlap_briks = briks_subjects_set & rois_subjects_set

print(f"\n{'='*60}")
print(f"Comparison: briks.txt vs habenula-rois subjects")
print('='*60)
print(f"Subjects in briks.txt but NOT in habenula-rois: {len(briks_only)}")
if len(briks_only) > 0:
    print(sorted(briks_only)[:20])  # Show first 20

print(f"\nSubjects in habenula-rois but NOT in briks.txt: {len(rois_only)}")
if len(rois_only) > 0:
    print(sorted(rois_only)[:20])  # Show first 20

print(f"\nSubjects in both: {len(overlap_briks)}")

Found 1212 unique subjects in briks.txt

First 10 subjects:
['sub-0050004', 'sub-0050005', 'sub-0050006', 'sub-0050007', 'sub-0050008', 'sub-0050010', 'sub-0050011', 'sub-0050012', 'sub-0050013', 'sub-0050014']

Comparison: briks.txt vs habenula-rois subjects
Subjects in briks.txt but NOT in habenula-rois: 0

Subjects in habenula-rois but NOT in briks.txt: 270
['sub-0050102', 'sub-0050103', 'sub-0050104', 'sub-0050105', 'sub-0050107', 'sub-0050111', 'sub-0050112', 'sub-0050113', 'sub-0050114', 'sub-0050115', 'sub-0050116', 'sub-0050117', 'sub-0050119', 'sub-0050134', 'sub-0050143', 'sub-0050144', 'sub-0050146', 'sub-0050147', 'sub-0050149', 'sub-0050152']

Subjects in both: 1212


In [32]:
# List all subjects in habenula-rois that don't appear in briks.txt
print(f"Subjects with drawn habenulas NOT in briks.txt ({len(rois_only)} subjects):\n")
for subject in sorted(rois_only):
    print(subject)

Subjects with drawn habenulas NOT in briks.txt (270 subjects):

sub-0050102
sub-0050103
sub-0050104
sub-0050105
sub-0050107
sub-0050111
sub-0050112
sub-0050113
sub-0050114
sub-0050115
sub-0050116
sub-0050117
sub-0050119
sub-0050134
sub-0050143
sub-0050144
sub-0050146
sub-0050147
sub-0050149
sub-0050152
sub-0050159
sub-0050163
sub-0050164
sub-0050169
sub-0050291
sub-0050319
sub-0050361
sub-0050363
sub-0050370
sub-0050433
sub-0050434
sub-0050435
sub-0050436
sub-0050437
sub-0050438
sub-0050439
sub-0050440
sub-0050441
sub-0050442
sub-0050443
sub-0050444
sub-0050445
sub-0050446
sub-0050447
sub-0050448
sub-0050449
sub-0050463
sub-0050466
sub-0050467
sub-0050468
sub-0050469
sub-0050470
sub-0050477
sub-0050480
sub-0050482
sub-0050483
sub-0050485
sub-0050486
sub-0050487
sub-0050488
sub-0050490
sub-0050491
sub-0050492
sub-0050493
sub-0050494
sub-0050496
sub-0050497
sub-0050498
sub-0050499
sub-0050500
sub-0050501
sub-0050502
sub-0050503
sub-0050504
sub-0050507
sub-0050509
sub-0050514
sub-0050515
